# Camada Gold e Análise dos Dados

Nesta etapa, os dados tratados na camada Silver são agregados e integrados para responder às perguntas de negócio definidas para o projeto.

A camada Gold terá como foco a análise da relação entre a produção agrícola de soja e a cadeia produtiva do biodiesel no Brasil.

## Perguntas de negócio

1. Como evoluíram a produção de soja e a produção de biodiesel no Brasil ao longo do período analisado?
2. Como a produção de soja e a produção de biodiesel estão distribuídas entre as regiões brasileiras?
3. Como evoluiu a utilização de matérias-primas derivadas da soja na produção de biodiesel?
4. Existe associação entre a produção agrícola de soja e a produção de biodiesel ao longo do período analisado?

In [0]:
# Carregamento das três Silver

from pyspark.sql import functions as F

df_soja = spark.table("silver_ibge_soja")
df_biodiesel = spark.table("silver_anp_biodiesel")
df_materia_prima = spark.table("silver_anp_materia_prima")

print("IBGE - Soja:", df_soja.count())
print("ANP - Biodiesel:", df_biodiesel.count())
print("ANP - Matérias-primas:", df_materia_prima.count())

IBGE - Soja: 243
ANP - Biodiesel: 11340
ANP - Matérias-primas: 4745


## Agregação dos dados de produção de soja

Os dados do IBGE estão disponíveis na camada Silver com granularidade de Unidade da Federação e ano.

Para permitir a integração com os dados de produção de biodiesel, os registros são agregados por região e ano. A produção de soja é obtida pela soma da quantidade produzida pelas UFs de cada região.

Os valores ausentes permanecem nulos na camada Silver e não são substituídos por zero, evitando assumir ausência de produção quando a fonte indica ausência de informação.

In [0]:
# Agregação da produção de soja por região e ano

df_soja_regiao_ano = (
    df_soja
    .groupBy("ano", "regiao")
    .agg(
        F.sum("producao_soja_t").alias("producao_soja_t")
    )
    .orderBy("ano", "regiao")
)

print("Registros soja por região/ano:", df_soja_regiao_ano.count())

display(df_soja_regiao_ano)

Registros soja por região/ano: 45


ano,regiao,producao_soja_t
2015,CENTRO-OESTE,4.3943604E7
2015,NORDESTE,8386412.0
2015,NORTE,4274638.0
2015,SUDESTE,5930317.0
2015,SUL,3.4929965E7
2016,CENTRO-OESTE,4.4140654E7
2016,NORDESTE,5145197.0
2016,NORTE,4096882.0
2016,SUDESTE,7540290.0
2016,SUL,3.5471797E7


## Agregação dos dados de produção de biodiesel

Os dados de produção de biodiesel da ANP na camada Silver possuem granularidade mensal e por produtor.

Para permitir a integração com os dados agrícolas do IBGE, a produção de biodiesel é agregada por região e ano, somando os volumes produzidos pelos diferentes produtores e meses.

A agregação estabelece uma granularidade comum de região e ano entre as duas fontes.

In [0]:
# Agregação da produção de biodiesel por região e ano

df_biodiesel_regiao_ano = (
    df_biodiesel
    .groupBy("ano", "regiao")
    .agg(
        F.sum("producao_biodiesel").alias("producao_biodiesel_m3")
    )
    .orderBy("ano", "regiao")
)

print(
    "Registros biodiesel por região/ano:",
    df_biodiesel_regiao_ano.count()
)

display(df_biodiesel_regiao_ano)

Registros biodiesel por região/ano: 45


ano,regiao,producao_biodiesel_m3
2015,CENTRO-OESTE,1748407.0599999996
2015,NORDESTE,314716.53700000007
2015,NORTE,66224.751
2015,SUDESTE,295435.6940000001
2015,SUL,1512484.492
2016,CENTRO-OESTE,1646827.5500000005
2016,NORDESTE,304604.90200000006
2016,NORTE,38957.789000000004
2016,SUDESTE,254258.66600000003
2016,SUL,1556690.0909999995


## Integração da produção de soja e biodiesel

Após a agregação das duas fontes para a granularidade comum de região e ano, os dados de produção agrícola de soja do IBGE são integrados aos dados de produção de biodiesel da ANP.

A integração utiliza as colunas `ano` e `regiao` como chaves, permitindo comparar as duas medidas sem multiplicação indevida de registros causada pelas diferentes granularidades das fontes originais.

In [0]:
# Integração dos dados de soja e biodiesel por região e ano

df_gold_soja_biodiesel = (
    df_soja_regiao_ano
    .join(
        df_biodiesel_regiao_ano,
        on=["ano", "regiao"],
        how="inner"
    )
    .withColumn(
        "producao_soja_t",
        F.round(F.col("producao_soja_t"), 2)
    )
    .withColumn(
        "producao_biodiesel_m3",
        F.round(F.col("producao_biodiesel_m3"), 2)
    )
    .orderBy("ano", "regiao")
)

print(
    "Registros Gold soja + biodiesel:",
    df_gold_soja_biodiesel.count()
)

display(df_gold_soja_biodiesel)

Registros Gold soja + biodiesel: 45


ano,regiao,producao_soja_t,producao_biodiesel_m3
2015,CENTRO-OESTE,4.3943604E7,1748407.06
2015,NORDESTE,8386412.0,314716.54
2015,NORTE,4274638.0,66224.75
2015,SUDESTE,5930317.0,295435.69
2015,SUL,3.4929965E7,1512484.49
2016,CENTRO-OESTE,4.4140654E7,1646827.55
2016,NORDESTE,5145197.0,304604.9
2016,NORTE,4096882.0,38957.79
2016,SUDESTE,7540290.0,254258.67
2016,SUL,3.5471797E7,1556690.09


## Matérias-primas derivadas da soja

Para analisar a utilização da soja na cadeia produtiva do biodiesel, são selecionadas as matérias-primas cujo nome contém o termo `SOJA`.

A validação realizada na camada Silver identificou duas categorias: `ÓLEO DE SOJA (GLYCINE MAX)` e `ÁCIDO GRAXO DE ÓLEO DE SOJA`.

Os volumes são agregados por região e ano. Essa fonte possui dados entre janeiro de 2017 e agosto de 2023, portanto sua cobertura temporal é diferente das bases de produção agrícola e de biodiesel.

In [0]:
# Agregação das matérias-primas derivadas da soja por região e ano

df_gold_materia_prima_soja = (
    df_materia_prima
    .filter(F.upper(F.col("produto")).contains("SOJA"))
    .groupBy("ano", "regiao")
    .agg(
        F.sum("quantidade_m3").alias("materia_prima_soja_m3")
    )
    .withColumn(
        "materia_prima_soja_m3",
        F.round(F.col("materia_prima_soja_m3"), 2)
    )
    .orderBy("ano", "regiao")
)

print(
    "Registros Gold matérias-primas de soja:",
    df_gold_materia_prima_soja.count()
)

display(df_gold_materia_prima_soja)

Registros Gold matérias-primas de soja: 34


ano,regiao,materia_prima_soja_m3
2017,CENTRO-OESTE,1268646.0
2017,NORDESTE,142851.0
2017,SUDESTE,86663.0
2017,SUL,1273535.0
2018,CENTRO-OESTE,1723496.0
2018,NORDESTE,144655.0
2018,NORTE,66602.0
2018,SUDESTE,107668.0
2018,SUL,1647022.0
2019,CENTRO-OESTE,1926135.0


In [0]:
# Persistência das tabelas da camada Gold

df_gold_soja_biodiesel.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_soja_biodiesel_regiao_ano")

df_gold_materia_prima_soja.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_materia_prima_soja_regiao_ano")

print("Tabelas Gold persistidas com sucesso.")

Tabelas Gold persistidas com sucesso.


In [0]:
# Validação das tabelas Gold persistidas

spark.sql("""
    SELECT
        'gold_soja_biodiesel_regiao_ano' AS tabela,
        COUNT(*) AS registros
    FROM gold_soja_biodiesel_regiao_ano

    UNION ALL

    SELECT
        'gold_materia_prima_soja_regiao_ano',
        COUNT(*)
    FROM gold_materia_prima_soja_regiao_ano
""").show(truncate=False)

+----------------------------------+---------+
|tabela                            |registros|
+----------------------------------+---------+
|gold_soja_biodiesel_regiao_ano    |45       |
|gold_materia_prima_soja_regiao_ano|34       |
+----------------------------------+---------+



# Qualidade dos Dados

A qualidade dos dados foi avaliada considerando completude, consistência, unicidade e presença de valores potencialmente atípicos.

As verificações foram realizadas sobre os dados tratados e sobre as tabelas analíticas utilizadas nas análises, buscando identificar limitações que possam afetar a interpretação dos resultados.

In [0]:
# Verificações de qualidade das tabelas Gold

qualidade_gold = spark.sql("""
    SELECT
        'gold_soja_biodiesel_regiao_ano' AS tabela,
        COUNT(*) AS registros,
        SUM(CASE WHEN ano IS NULL THEN 1 ELSE 0 END) AS ano_nulo,
        SUM(CASE WHEN regiao IS NULL THEN 1 ELSE 0 END) AS regiao_nula,
        SUM(CASE WHEN producao_soja_t IS NULL THEN 1 ELSE 0 END) AS soja_nula,
        SUM(CASE WHEN producao_biodiesel_m3 IS NULL THEN 1 ELSE 0 END) AS biodiesel_nulo
    FROM gold_soja_biodiesel_regiao_ano
""")

display(qualidade_gold)

tabela,registros,ano_nulo,regiao_nula,soja_nula,biodiesel_nulo
gold_soja_biodiesel_regiao_ano,45,0,0,0,0


In [0]:
# Verificação de unicidade e consistência da Gold principal

qualidade_consistencia = spark.sql("""
    SELECT
        COUNT(*) AS total_registros,
        COUNT(DISTINCT CONCAT(ano, '-', regiao)) AS chaves_unicas,
        SUM(CASE WHEN producao_soja_t < 0 THEN 1 ELSE 0 END) AS soja_negativa,
        SUM(CASE WHEN producao_biodiesel_m3 < 0 THEN 1 ELSE 0 END) AS biodiesel_negativo,
        COUNT(DISTINCT ano) AS quantidade_anos,
        COUNT(DISTINCT regiao) AS quantidade_regioes
    FROM gold_soja_biodiesel_regiao_ano
""")

display(qualidade_consistencia)

total_registros,chaves_unicas,soja_negativa,biodiesel_negativo,quantidade_anos,quantidade_regioes
45,45,0,0,9,5


In [0]:
# Estatísticas descritivas das medidas da Gold principal

estatisticas_gold = (
    spark.table("gold_soja_biodiesel_regiao_ano")
    .select(
        "producao_soja_t",
        "producao_biodiesel_m3"
    )
    .summary(
        "count",
        "min",
        "25%",
        "50%",
        "75%",
        "max"
    )
)

display(estatisticas_gold)

summary,producao_soja_t,producao_biodiesel_m3
count,45,45
min,4096882.0,7821.29
25%,8511678.0,295435.69
50%,1.2118865E7,478223.92
75%,3.8912491E7,2199033.18
max,7.6330093E7,3182137.56


### Avaliação de valores extremos

As estatísticas descritivas evidenciam uma amplitude elevada tanto na produção de soja quanto na produção de biodiesel entre as observações de região e ano.

Essa variação é compatível com as diferenças existentes entre as regiões brasileiras quanto à produção agrícola e à produção de biocombustíveis. Por esse motivo, os valores extremos foram mantidos, uma vez que representam observações válidas das fontes e não foram identificados indícios de erro que justificassem sua exclusão.

Dessa forma, não foi realizado tratamento por remoção ou substituição de valores extremos.

# Análise de Dados

Nesta etapa, os dados disponibilizados na camada Gold são utilizados para responder às perguntas de negócio definidas para o projeto.

As análises consideram a evolução temporal e a distribuição regional da produção de soja e de biodiesel, a utilização de matérias-primas derivadas da soja e a associação entre as duas cadeias produtivas.

## Pergunta 1 — Evolução da produção de soja e biodiesel

**Como evoluíram a produção de soja e a produção de biodiesel no Brasil ao longo do período analisado?**

Para analisar a evolução nacional, os dados da tabela Gold, originalmente organizados por região e ano, são agregados por ano. Dessa forma, são obtidos os volumes anuais de produção de soja e de biodiesel no Brasil entre 2015 e 2023.

In [0]:
from pyspark.sql import functions as F

# Agregação nacional da produção de soja e biodiesel por ano

df_evolucao_nacional = (
    spark.table("gold_soja_biodiesel_regiao_ano")
    .groupBy("ano")
    .agg(
        F.sum("producao_soja_t").alias("producao_soja_t"),
        F.sum("producao_biodiesel_m3").alias("producao_biodiesel_m3")
    )
    .withColumn(
        "producao_soja_t",
        F.round(F.col("producao_soja_t"), 2)
    )
    .withColumn(
        "producao_biodiesel_m3",
        F.round(F.col("producao_biodiesel_m3"), 2)
    )
    .orderBy("ano")
)

print("Quantidade de anos:", df_evolucao_nacional.count())

display(df_evolucao_nacional)

Quantidade de anos: 9


ano,producao_soja_t,producao_biodiesel_m3
2015,9.7464936E7,3937268.53
2016,9.639482E7,3801339.0
2017,1.14732101E8,4289839.69
2018,1.1791245E8,5336635.06
2019,1.14316829E8,5902787.8
2020,1.21796549E8,6445179.78
2021,1.34468168E8,6770661.06
2022,1.21290103E8,6254710.27
2023,1.52144238E8,7532435.7


### Evolução relativa das duas séries

Como a produção de soja é medida em toneladas e a produção de biodiesel em metros cúbicos, os valores absolutos não são diretamente comparáveis.

Para permitir a comparação da evolução temporal das duas séries, os valores foram transformados em índices com base 100 em 2015. Dessa forma, o gráfico representa a variação relativa de cada produção em relação ao primeiro ano do período analisado.

In [0]:
# Criação de índices base 100 para comparação da evolução temporal

valores_2015 = (
    df_evolucao_nacional
    .filter(F.col("ano") == 2015)
    .select("producao_soja_t", "producao_biodiesel_m3")
    .first()
)

soja_base = valores_2015["producao_soja_t"]
biodiesel_base = valores_2015["producao_biodiesel_m3"]

df_evolucao_indice = (
    df_evolucao_nacional
    .withColumn(
        "indice_soja",
        F.round(F.col("producao_soja_t") / F.lit(soja_base) * 100, 2)
    )
    .withColumn(
        "indice_biodiesel",
        F.round(F.col("producao_biodiesel_m3") / F.lit(biodiesel_base) * 100, 2)
    )
    .select(
        "ano",
        "indice_soja",
        "indice_biodiesel"
    )
    .orderBy("ano")
)

display(df_evolucao_indice)

ano,indice_soja,indice_biodiesel
2015,100.0,100.0
2016,98.9,96.55
2017,117.72,108.95
2018,120.98,135.54
2019,117.29,149.92
2020,124.96,163.7
2021,137.97,171.96
2022,124.44,158.86
2023,156.1,191.31


Databricks visualization. Run in Databricks to view.

**Interpretação:** considerando 2015 como ano-base (índice = 100), observa-se crescimento tanto da produção de soja quanto da produção de biodiesel no Brasil ao longo do período analisado. Em 2023, o índice da produção de soja atingiu 156,1, enquanto o índice da produção de biodiesel alcançou 191,31.

Isso representa um crescimento aproximado de 56% na produção de soja e de 91% na produção de biodiesel em relação a 2015. Embora as duas séries apresentem tendência geral de crescimento, a produção de biodiesel apresentou expansão relativa mais intensa no período, especialmente a partir de 2018.

As oscilações observadas em alguns anos mostram, entretanto, que as duas produções não evoluem de forma proporcional, aspecto que será explorado posteriormente na análise de associação entre as variáveis.

## 2. Distribuição regional da produção de soja e biodiesel

Para analisar a distribuição regional das duas cadeias produtivas, os dados da tabela Gold são agregados por região considerando todo o período de 2015 a 2023.

Como as produções de soja e biodiesel são expressas em unidades diferentes, toneladas e metros cúbicos, respectivamente, sua distribuição será analisada separadamente entre as regiões brasileiras.

In [0]:
# Distribuição regional da produção de soja e biodiesel no período

df_distribuicao_regional = (
    spark.table("gold_soja_biodiesel_regiao_ano")
    .groupBy("regiao")
    .agg(
        F.sum("producao_soja_t").alias("producao_soja_t"),
        F.sum("producao_biodiesel_m3").alias("producao_biodiesel_m3")
    )
    .withColumn(
        "producao_soja_t",
        F.round(F.col("producao_soja_t"), 2)
    )
    .withColumn(
        "producao_biodiesel_m3",
        F.round(F.col("producao_biodiesel_m3"), 2)
    )
    .orderBy(F.desc("producao_soja_t"))
)

display(df_distribuicao_regional)

regiao,producao_soja_t,producao_biodiesel_m3
CENTRO-OESTE,5.03537196E8,2.04696473E7
SUL,3.25340723E8,2.115199168E7
NORDESTE,9.8093644E7,3932580.37
SUDESTE,8.6396974E7,3654862.81
NORTE,5.7151657E7,1061774.73


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

**Interpretação:** a distribuição regional evidencia diferenças entre a produção agrícola de soja e a produção de biodiesel no Brasil. No período de 2015 a 2023, o Centro-Oeste concentrou a maior produção acumulada de soja, seguido pela região Sul, enquanto Nordeste, Sudeste e Norte apresentaram volumes consideravelmente menores.

Na produção de biodiesel, entretanto, a distribuição apresenta comportamento diferente. Sul e Centro-Oeste concentram os maiores volumes acumulados, com a região Sul apresentando produção ligeiramente superior à do Centro-Oeste.

Os resultados indicam que uma maior produção regional de soja não se traduz, necessariamente, em uma produção de biodiesel proporcionalmente maior. Essa diferença reforça a necessidade de analisar separadamente a disponibilidade agrícola da soja e sua utilização na cadeia produtiva do biodiesel.

## 3. Utilização de matérias-primas derivadas da soja

Para analisar a evolução da utilização de matérias-primas derivadas da soja na produção de biodiesel, são utilizados os dados da ANP consolidados na tabela Gold `gold_materia_prima_soja_regiao_ano`.

Os dados disponíveis para matérias-primas abrangem o período de janeiro de 2017 a agosto de 2023. Dessa forma, a análise da evolução anual considera inicialmente os anos completos de 2017 a 2022, evitando a comparação direta de 2023, que possui apenas dados parciais.

In [0]:
# Evolução nacional do uso de matérias-primas derivadas da soja

df_materia_prima_nacional = (
    spark.table("gold_materia_prima_soja_regiao_ano")
    .filter(F.col("ano").between(2017, 2022))
    .groupBy("ano")
    .agg(
        F.sum("materia_prima_soja_m3").alias("materia_prima_soja_m3")
    )
    .withColumn(
        "materia_prima_soja_m3",
        F.round(F.col("materia_prima_soja_m3"), 2)
    )
    .orderBy("ano")
)

display(df_materia_prima_nacional)

ano,materia_prima_soja_m3
2017,2771695.0
2018,3689443.0
2019,4094183.0
2020,4690549.0
2021,4932399.0
2022,4234156.0


Databricks visualization. Run in Databricks to view.

**Interpretação:** entre 2017 e 2021, observa-se crescimento contínuo na utilização de matérias-primas derivadas da soja na produção de biodiesel, passando de aproximadamente 2,77 milhões de m³ para 4,93 milhões de m³. Em 2022, entretanto, o volume recuou para aproximadamente 4,23 milhões de m³.

Apesar da redução observada em 2022, o volume permaneceu superior ao registrado no início da série, indicando expansão da utilização de matérias-primas derivadas da soja ao longo do período analisado.

O ano de 2023 não foi incluído nesta comparação anual, pois os dados disponíveis abrangem apenas o período de janeiro a agosto, o que impediria uma comparação direta com os anos completos anteriores.

## 4. Associação entre a produção de soja e a produção de biodiesel

Para investigar a relação entre as duas variáveis, é calculado o coeficiente de correlação de Pearson entre a produção anual de soja e a produção anual de biodiesel no Brasil entre 2015 e 2023.

A análise possui caráter descritivo e busca identificar se as duas séries apresentam associação ao longo do período. A existência de correlação não implica relação de causalidade entre a produção agrícola de soja e a produção de biodiesel.

In [0]:
# Correlação entre a produção nacional de soja e biodiesel

correlacao = df_evolucao_nacional.stat.corr(
    "producao_soja_t",
    "producao_biodiesel_m3"
)

print(f"Correlação de Pearson: {correlacao:.4f}")

Correlação de Pearson: 0.9131


In [0]:
# Dados para visualização da associação entre soja e biodiesel

df_associacao = (
    df_evolucao_nacional
    .select(
        "ano",
        "producao_soja_t",
        "producao_biodiesel_m3"
    )
    .orderBy("ano")
)

display(df_associacao)

ano,producao_soja_t,producao_biodiesel_m3
2015,9.7464936E7,3937268.53
2016,9.639482E7,3801339.0
2017,1.14732101E8,4289839.69
2018,1.1791245E8,5336635.06
2019,1.14316829E8,5902787.8
2020,1.21796549E8,6445179.78
2021,1.34468168E8,6770661.06
2022,1.21290103E8,6254710.27
2023,1.52144238E8,7532435.7


Databricks visualization. Run in Databricks to view.

**Interpretação:** o coeficiente de correlação de Pearson calculado para o período de 2015 a 2023 foi de aproximadamente 0,91, indicando uma associação linear positiva elevada entre a produção nacional de soja e a produção nacional de biodiesel.

O gráfico de dispersão reforça esse resultado, mostrando que anos com maiores volumes de produção de soja tendem a estar associados a maiores volumes de produção de biodiesel.

Entretanto, essa associação não deve ser interpretada como relação de causalidade. A produção de biodiesel depende de outros fatores além da disponibilidade agrícola da soja, e a análise considera apenas nove observações anuais. Além disso, ambas as séries apresentam tendência de crescimento ao longo do período, o que também pode contribuir para a correlação observada.

Portanto, os resultados indicam associação entre as duas séries no período analisado, mas não permitem concluir que o crescimento da produção de soja seja responsável pelo crescimento da produção de biodiesel.

In [0]:
# Schemas das tabelas Gold

spark.table("gold_soja_biodiesel_regiao_ano").printSchema()
spark.table("gold_materia_prima_soja_regiao_ano").printSchema()

root
 |-- ano: integer (nullable = true)
 |-- regiao: string (nullable = true)
 |-- producao_soja_t: double (nullable = true)
 |-- producao_biodiesel_m3: double (nullable = true)

root
 |-- ano: integer (nullable = true)
 |-- regiao: string (nullable = true)
 |-- materia_prima_soja_m3: double (nullable = true)

